# FestiVox — Group-12, Assignment-2
Vector database (ChromaDB) + semantic search + RAG over attendee feedback for the Assignment-1 Music Festival system.

Prerequisites: `pip install -r requirements.txt`, Ollama running with `llama3.2:3b`.

## Part A — dataset, embeddings, ChromaDB, CRUD

In [ ]:
!python generate_dataset.py
import pandas as pd
df = pd.read_csv('data/festival_feedback.csv', keep_default_na=False)
print(df.shape); df.sample(5, random_state=1)

In [ ]:
!python part_a_build.py
from common import load
a = load('part_a.json'); a['crud']['after_update']

## Part B — semantic search with configurable K and metadata filters

In [ ]:
from common import Embedder, get_collection
from part_b_search import semantic_search
emb = Embedder(); _, col = get_collection()
K = 5  # configurable
r = semantic_search(col, emb, 'Complaints that the music was so loud it hurt people\'s ears', K,
                    {'$and': [{'genre': 'EDM'}, {'rating': {'$lte': 2}}]})
list(zip(r['ids'][0], [round(d,4) for d in r['distances'][0]], r['documents'][0]))

In [ ]:
!python part_b_search.py 5
b = load('part_b.json')
import pandas as pd
pd.DataFrame([{ 'qid':q['qid'], 'student':q['student'], 'FR keyword':q['fact_recall_keyword'], 'FR semantic':q['fact_recall_semantic'], 'FR filtered':q['fact_recall_filtered'], **{f'{m}@10':q['metrics'][f'raw|k10|{m}']['fact_recall'] for m in ['l2','cosine','dot']}, 'normalised identical':q['overlap']['normalized|k10']['identical_order_all3']} for q in b['queries']])

In [ ]:
# exhaustive search, one query, all three metrics, K=10
q = b['queries'][7]
pd.DataFrame({m:[f"{h['id']} ({h['score']})" for h in q['metrics'][f'raw|k10|{m}']['hits']] for m in ['l2','cosine','dot']})

## Part C — RAG with Ollama (llama3.2:3b)

In [ ]:
!python part_c_rag.py ollama
c = load('part_c_ollama.json')
for r in c['queries']:
    print('='*100); print(r['qid'], r['text']); print('retrieved:', [x['id'] for x in r['retrieved']])
    print('RAG:', r['answer']); print('--- no context:', r['no_context_answer'][:500])

## Report

In [ ]:
!python build_report.py